# CoRe-TFM — JMLR Robustness EXECUTABLE V2.1

This replaces the buggy V2 orchestration while preserving the original Q1 inference implementation.

**What V2.1 fixes**
- freezes one source commit and reuses it on every resume;
- uses a fresh Drive evidence root so old V2 partial outputs cannot contaminate corrected runs;
- statically checks every original Q1 engine cell through 12E before GPU work;
- patches the actual shard time budget as well as the protocol time budget;
- tolerates missing/empty partial result files safely;
- reports recorded fold failures directly;
- creates aggregation folders safely during partial progress; and
- uses the frozen rare-class thresholds `[1, 2, 5, 10]`.

Run all. One invocation works on the next incomplete real-TFM variant. Re-run the notebook to continue.


## 1. Mount Drive, select/freeze repository source, and install dependencies

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, traceback
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

ROOT = Path('/content/core-tfm')
ROBUST_ROOT = Path('/content/drive/MyDrive/CoRe_TFM_Q1/core_tfm_jmlr_robustness_v2_1')
ROBUST_ROOT.mkdir(parents=True, exist_ok=True)
PROTOCOL_PATH = ROBUST_ROOT/'ROBUSTNESS_PROTOCOL.json'

if not (ROOT/'.git').exists():
    subprocess.run(['git','clone','https://github.com/bnssaanirudh/core-tfm.git',str(ROOT)], check=True)
subprocess.run(['git','fetch','origin'], cwd=ROOT, check=True)

if PROTOCOL_PATH.exists():
    existing_protocol = json.loads(PROTOCOL_PATH.read_text())
    SOURCE_COMMIT = existing_protocol['source_commit_at_freeze']
    print('Resuming frozen source:', SOURCE_COMMIT)
else:
    SOURCE_COMMIT = subprocess.check_output(
        ['git','rev-parse','origin/main'], cwd=ROOT, text=True
    ).strip()
    print('New robustness source candidate:', SOURCE_COMMIT)

subprocess.run(['git','checkout','--detach',SOURCE_COMMIT], cwd=ROOT, check=True)
HEAD = subprocess.check_output(['git','rev-parse','HEAD'], cwd=ROOT, text=True).strip()
assert HEAD == SOURCE_COMMIT

subprocess.run([
    sys.executable,'-m','pip','install','-q','-e','.[test]',
    'pyyaml','ucimlrepo','hf_transfer'
], cwd=ROOT, check=True)

SRC = str(ROOT/'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
os.chdir(ROOT)

from core_tfm.robustness_runner import (
    load_notebook, patch_q1_notebook, code_cells_through_shard_12e,
    fold_result_status, write_complete_marker,
)

print('Repository:', ROOT)
print('Frozen HEAD:', HEAD)
print('Robustness root:', ROBUST_ROOT)


## 2. Load the frozen experiment specification and freeze/verify the protocol

In [ ]:
import yaml
cfg = yaml.safe_load((ROOT/'configs'/'reliability_aware_experiments.yaml').read_text())

protocol = {
    'run_id': 'core_tfm_jmlr_robustness_v2_1',
    'source_commit_at_freeze': HEAD,
    'inference_engine': 'notebooks/CoRe_TFM_Q1_FAST_COMPLETE_256_Colab.ipynb',
    'multi_seed': cfg['seed_robustness'],
    'context_size': cfg['context_size'],
    'safe_selective': cfg['safe_selective'],
    'rare_class_sensitivity': cfg['rare_class_sensitivity'],
    'primary_models': cfg['models']['primary'],
    'boundary_models': cfg['models'].get('boundary', []),
    'outcome_blind_freeze': True,
}

if PROTOCOL_PATH.exists():
    frozen_protocol = json.loads(PROTOCOL_PATH.read_text())
    assert frozen_protocol['source_commit_at_freeze'] == HEAD, (
        'Protocol/source mismatch. Do not mix evidence from different commits.'
    )
    for key in ['multi_seed','context_size','safe_selective','rare_class_sensitivity','primary_models']:
        assert frozen_protocol[key] == protocol[key], f'Frozen protocol mismatch in {key}'
    print('Existing robustness protocol verified.')
else:
    PROTOCOL_PATH.write_text(json.dumps(protocol, indent=2))
    frozen_protocol = protocol
    print('Robustness protocol frozen.')

display(frozen_protocol)


## 3. Static preflight of the original Q1 inference engine — no TFM inference yet

In [ ]:
TEMPLATE_PATH = ROOT/'notebooks'/'CoRe_TFM_Q1_FAST_COMPLETE_256_Colab.ipynb'
template = load_notebook(TEMPLATE_PATH)
preflight_nb = patch_q1_notebook(
    template,
    run_id='_static_preflight/seed_11',
    seed=11, train_limit=256, test_limit=128,
    drive_base=str(ROBUST_ROOT),
    session_minutes=25, shard_minutes=25,
    disable_controlled_replications=True,
    disable_selection_ablations=True,
    disable_validation_sensitivity=True,
)
preflight_sources = code_cells_through_shard_12e(preflight_nb)
assert len(preflight_sources) >= 10, 'Unexpectedly few Q1 engine cells found.'
print(f'STATIC ENGINE PREFLIGHT PASS: {len(preflight_sources)} Python cells compile through 12E.')


## 4. Select the next incomplete real-TFM variant

Priority: all five sampling seeds first, then the 15 context-size variants. Status includes any recorded fold failure from an earlier attempt.


In [ ]:
seed_variants = [
    {
        'group':'multi_seed',
        'seed':int(seed),
        'train_limit':int(cfg['seed_robustness']['train_limit']),
        'test_limit':int(cfg['seed_robustness']['test_limit']),
        'name':f'seed_{seed}',
    }
    for seed in cfg['seed_robustness']['seeds']
]
context_variants = [
    {
        'group':'context_size',
        'seed':int(seed),
        'train_limit':int(size),
        'test_limit':int(cfg['context_size'].get('test_limit',128)),
        'name':f'seed_{seed}_train_{size}',
    }
    for seed in cfg['context_size']['seeds']
    for size in cfg['context_size']['train_sizes']
]
QUEUE = seed_variants + context_variants

status_rows = []
NEXT = None
for v in QUEUE:
    run_dir = ROBUST_ROOT/v['group']/v['name']
    st = fold_result_status(run_dir)
    status_rows.append({
        **v,
        'complete':st.get('complete',False),
        'rows':st.get('rows',0),
        'fold_cells':st.get('fold_cells',0),
        'failure_count':st.get('failure_count',0),
        'reason':st.get('reason'),
    })
    if NEXT is None and not st.get('complete',False):
        NEXT = v

status_df = pd.DataFrame(status_rows)
display(status_df)
print('NEXT VARIANT:', NEXT)
if NEXT is not None:
    next_status = fold_result_status(ROBUST_ROOT/NEXT['group']/NEXT['name'])
    if next_status.get('last_failure'):
        print('LAST RECORDED FAILURE FOR NEXT VARIANT:')
        print(json.dumps(next_status['last_failure'], indent=2))


## 5. Execute/resume exactly one real-TFM variant

The original Q1 code is executed unchanged except for the frozen run ID, sampling seed, train/test limits, output root, and time-budget/post-processing flags. Fold checkpoints are persisted after every completed task.


In [ ]:
if NEXT is None:
    print('All multi-seed and context-size real-TFM variants are structurally complete.')
else:
    template = load_notebook(TEMPLATE_PATH)
    patched = patch_q1_notebook(
        template,
        run_id=f"{NEXT['group']}/{NEXT['name']}",
        seed=NEXT['seed'],
        train_limit=NEXT['train_limit'],
        test_limit=NEXT['test_limit'],
        drive_base=str(ROBUST_ROOT),
        session_minutes=25,
        shard_minutes=25,
        disable_controlled_replications=True,
        disable_selection_ablations=True,
        disable_validation_sensitivity=True,
    )
    sources = code_cells_through_shard_12e(patched)
    print(f"Executing {len(sources)} Q1 engine cells for {NEXT}")
    engine_ns = {'__name__':'__robustness_exec__'}

    for idx, source in enumerate(sources, 1):
        first = source.lstrip().splitlines()[0] if source.strip() else '<empty>'
        print(f'--- Q1 engine {idx}/{len(sources)}: {first} ---', flush=True)
        try:
            exec(compile(source, f'<q1_engine_cell_{idx}>', 'exec'), engine_ns, engine_ns)
        except Exception:
            print(f'ENGINE CELL FAILED: {idx} :: {first}')
            traceback.print_exc()
            raise

    run_dir = ROBUST_ROOT/NEXT['group']/NEXT['name']
    st = fold_result_status(run_dir)
    print('VARIANT STATUS:')
    print(json.dumps(st, indent=2))

    if st.get('complete'):
        marker = write_complete_marker(run_dir, {
            'group':NEXT['group'],
            'seed':NEXT['seed'],
            'requested_train_limit':NEXT['train_limit'],
            'requested_test_limit':NEXT['test_limit'],
            'source_commit':HEAD,
            'inference_engine':'original_Q1_notebook_through_12E',
        })
        print('VARIANT COMPLETE:', marker)
    else:
        print('Variant is still incomplete. Re-run this notebook to resume it.')
        if st.get('last_failure'):
            print('Last recorded fold failure:')
            print(json.dumps(st['last_failure'], indent=2))


## 6. Aggregate partial/completed real-TFM progress safely

In [ ]:
rare_thresholds = cfg['rare_class_sensitivity']['minimum_support_thresholds']
subprocess.run([
    sys.executable, str(ROOT/'experiments'/'summarize_robustness_runs.py'),
    '--root', str(ROBUST_ROOT),
    '--seeds', ','.join(map(str,cfg['seed_robustness']['seeds'])),
    '--context-seeds', ','.join(map(str,cfg['context_size']['seeds'])),
    '--context-sizes', ','.join(map(str,cfg['context_size']['train_sizes'])),
    '--rare-thresholds', ','.join(map(str,rare_thresholds)),
], cwd=ROOT, check=True)

progress = json.loads((ROBUST_ROOT/'ROBUSTNESS_STATUS.json').read_text())
display(pd.DataFrame([
    {
        'group':name,
        'complete':payload.get('complete',False),
        'completed_variants':payload.get('completed_variants'),
        'expected_variants':payload.get('expected_variants'),
        'reason':payload.get('reason'),
    }
    for name,payload in progress.items()
]))


## 7. Controlled Safe Selective CoRe

This is a known-truth controlled experiment. It is not presented as a fresh real-TFM result. It saves all validation candidate scores and freezes selection before exact test scoring.


In [ ]:
SAFE_DIR = ROBUST_ROOT/'safe_selective'
SAFE_DIR.mkdir(parents=True, exist_ok=True)
if not (SAFE_DIR/'COMPLETE.json').exists():
    subprocess.run([
        sys.executable, str(ROOT/'experiments'/'run_safe_selective_controlled.py'),
        '--tasks','100',
        '--beta','1.0',
        '--delta',str(cfg['safe_selective']['delta']),
        '--output-dir',str(SAFE_DIR),
    ], cwd=ROOT, check=True)
safe_complete = SAFE_DIR/'COMPLETE.json'
assert safe_complete.exists() and safe_complete.stat().st_size > 0
display(json.loads(safe_complete.read_text()))


## 8. Archive-derived view-reliability diagnostic

This is explicitly a proxy. The frozen Q1 table does not contain the full fresh per-example direct-marginal archive required for the final JMLR reliability claim.


In [ ]:
VIEW_DIR = ROBUST_ROOT/'view_reliability'
VIEW_DIR.mkdir(parents=True, exist_ok=True)
derived = ROOT/'results'/'reliability_aware_v1'
derived.mkdir(parents=True, exist_ok=True)
subprocess.run([
    sys.executable, str(ROOT/'experiments'/'run_reliability_aware_suite.py'),
    '--fold-results', str(ROOT/'results'/'q1_fast_complete_256_v1'/'fold_results.csv'),
    '--output', str(derived),
], cwd=ROOT, check=True)

proxy_files = [
    'model_view_reliability_proxy.csv',
    'inconsistency_vs_gain.csv',
    'inconsistency_vs_gain_correlations.json',
]
for name in proxy_files:
    src = derived/name
    assert src.exists() and src.stat().st_size > 0, f'Missing proxy output: {src}'
    shutil.copy2(src, VIEW_DIR/name)

proxy_marker = {
    'complete':True,
    'scope':'archive_derived_proxy_only',
    'fresh_per_example_direct_marginal_archive':False,
    'claim_guard':'Diagnostic only; do not describe as completed fresh per-view TFM reliability reruns.',
}
(VIEW_DIR/'PROXY_COMPLETE.json').write_text(json.dumps(proxy_marker, indent=2))
display(proxy_marker)


## 9. Final gates

The core gate covers the experiments this notebook can truthfully complete. The full JMLR gate remains stricter and additionally requires a fresh per-example view-reliability archive and a successfully preflighted third TFM.


In [ ]:
def marker(group, filename='COMPLETE.json'):
    p = ROBUST_ROOT/group/filename
    return p.exists() and p.stat().st_size > 0

core_status = {
    'multi_seed': marker('multi_seed'),
    'context_size': marker('context_size'),
    'rare_class_exclusion': marker('rare_class'),
    'safe_selective_controlled': marker('safe_selective'),
    'view_reliability_proxy': marker('view_reliability','PROXY_COMPLETE.json'),
}
full_jmlr_status = {
    **core_status,
    'fresh_per_example_view_reliability': marker('view_reliability','FRESH_COMPLETE.json'),
    'third_tfm': marker('third_tfm'),
}

print('CORE STATUS:', core_status)
print('CORE ROBUSTNESS GATE:', 'PASS' if all(core_status.values()) else 'PENDING')
print('FULL JMLR STATUS:', full_jmlr_status)
print('FULL JMLR GATE:', 'PASS' if all(full_jmlr_status.values()) else 'PENDING')

if not all(core_status.values()):
    print('Re-run this notebook. It will continue the next incomplete real-TFM variant.')
